# Импорт библиотек

In [31]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score


# Считывание данных

In [4]:
df = pd.read_csv('data/processed.csv')
df.shape

(1001, 213)

# Удаление невалидных экземляров

In [6]:
matching_instances = df[df['IC50, mM'] == df['CC50, mM']]
print(len(matching_instances.index))
df.drop(index=matching_instances.index, inplace=True)
df.shape

135


(866, 213)

# Очистка таргета от выбросов

In [8]:
df = df[df['SI'] < 500]
df.shape

(849, 213)

# Подготовка данных для эксперемента

In [10]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

In [11]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['SI'].apply(lambda v: 1 if v >= 8 else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (679, 98), (679,)
Test dataset size: (170, 98), (170,)


# Эксперемент с моделями

In [13]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
3,Random Forest,0.74,0.74,0.74,0.74,170.0,0.76
5,Gradient Boosting,0.74,0.74,0.74,0.74,170.0,0.79
7,CatBoost,0.74,0.75,0.75,0.75,170.0,0.78
4,XGBoost,0.73,0.73,0.73,0.73,170.0,0.80
8,HistGradientBoosting,0.73,0.74,0.74,0.74,170.0,0.77
2,KNeighbors,0.70,0.72,0.75,0.72,170.0,0.75
6,LightGBM,0.70,0.71,0.70,0.71,170.0,0.79
9,AdaBoost,0.70,0.70,0.70,0.70,170.0,0.75
0,Logistic Regression,0.68,0.68,0.68,0.68,170.0,0.71
1,Decision Tree,0.67,0.67,0.67,0.67,170.0,0.68


# Подбор гиперпараметров

In [39]:
param_dist = {
    'n_estimators': randint(50, 250),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2']
}
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

Лучшие параметры: {'max_depth': 19, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 9, 'n_estimators': 130}


In [44]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.74
ROC AUC Score: 0.76


,precision,recall,f1-score,support
0,0.745455,0.836735,0.788462,98.000000
1,0.733333,0.611111,0.666667,72.000000
accuracy,0.741176,0.741176,0.741176,0.741176
macro avg,0.739394,0.723923,0.727564,170.000000
weighted avg,0.740321,0.741176,0.736878,170.000000
